### Harness工程
> Harness工程更关注整体架构，这跟我最近所思考的术与道有些相似。回顾过往，我一直在追求某种确定性的解决办法或者方案，就像解题一样，但实际上不是的，最主要的是思考和想法，正如算法题和代码一样，代码应该跟随逻辑，而不是为了解决一道题目，逻辑对，follow就完事。因此，学习Harnees工程辅助核心项目架构思路，再进行细节性的代码补充，才应该是正常的工程逻辑。抓大放下，分解问题，逐步实施，这才是正确的生活、工作方法。

本项目代码库：https://github.com/shareAI-lab/learn-claude-code/blob/main/README-zh.md

2026年4月23日，正式开启项目流程，以此作为简历项目支撑之一。

---

#### Harness逻辑
Agent工程本质是单个的Agent_Loop加上思考、工具调用，最后得到相关结果。其中，最重要的模式就是agent_loop:


In [3]:

def agent_loop(messages, llm):
    while True:
        response = llm.invoke(messages)
        messages.append({"role" : "assistance",
                        "content": response.contents})
        
        if response.stop_reason != "tool_use":
            return

        results = []

        for block in response.content:
            if block.type == "tool_use":
                output = TOOL_HANDLERS[block.name](**block.input)
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })

        messages.append({"role": "user", "content": results})

### S01 Agent循环
> "One loop & Bash is all you need" -- 一个工具 + 一个循环 = 一个 Agent。

> Harness 层: 循环 -- 模型与真实世界的第一道连接。
问题：
语言模型能推理代码, 但碰不到真实世界 -- 不能读文件、跑测试、看报错。没有循环, 每次工具调用你都得手动把结果粘回去。你自己就是那个循环。
解决方案：
```bash
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until stop_reason != "tool_use")

In [5]:
def agent_loop(query):
    messages = [{"role": "user", "content": query}]
    while True:
        response = client.messages.create(
            model=MODEL, system=SYSTEM, messages=messages,
            tools=TOOLS, max_tokens=8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return

        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = run_bash(block.input["command"])
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })
        messages.append({"role": "user", "content": results})

In [18]:
import subprocess

result = subprocess.run(
['dir'],
shell=True,
capture_output=True,
encoding='gbk'
)

result.check_returncode()
print(result)

CompletedProcess(args=['dir'], returncode=0, stdout=' 驱动器 E 中的卷没有标签。\n 卷的序列号是 842D-9A6E\n\n e:\\DATA\\LLM\\APX-LLM-Notebook\\Harness 的目录\n\n2026/04/23  10:13    <DIR>          .\n2026/04/23  10:13    <DIR>          ..\n2026/04/23  10:13                 0 Harness.ipynb\n               1 个文件              0 字节\n               2 个目录 1,067,682,775,040 可用字节\n', stderr='')


ModuleNotFoundError: No module named 'anthropic'